# Notebook 07: Spatial Mapping — Per-Pixel Isotopic Analysis

This notebook demonstrates the end-to-end spatial mapping pipeline:
build a synthetic hyperspectral neutron image, then recover 2D isotopic
density maps using `nereids.spatial_map()` and `nereids.fit_roi()`.

## Prerequisites

```bash
pixi run build
```

## New Functions

| Function | Description |
|----------|-------------|
| `spatial_map()` | Per-pixel parallel fitting across a 3D image stack |
| `fit_roi()` | Fit averaged spectrum over a rectangular region of interest |
| `load_tiff_stack()` | Load a multi-frame TIFF into a 3D numpy array |
| `normalize()` | Convert raw counts to transmission with uncertainty |
| `tof_to_energy_centers()` | Convert TOF bin edges to energy bin centers |

In [ ]:
import nereids
import numpy as np
import matplotlib.pyplot as plt
import time

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Create Synthetic Imaging Data

We build a 20×20 pixel synthetic neutron transmission image with two isotopes:

| Region | U-235 density | Pu-241 density |
|--------|---------------|----------------|
| Background (outer) | 0.001 atoms/barn | 0 |
| Center (inner 10×10) | 0.001 atoms/barn | 0.0005 atoms/barn |

This simulates a sample where Pu-241 is present only in the center.

In [ ]:
# Load ENDF data
u235 = nereids.load_endf(92, 235)
pu241 = nereids.load_endf(94, 241)
print(u235)
print(pu241)

# Energy grid: 1-15 eV with 200 points
# This range captures strong resonances for both isotopes
energies = np.linspace(1.0, 15.0, 200)

In [ ]:
# Build ground-truth density maps
height, width = 20, 20
true_u235 = np.full((height, width), 0.001)  # uniform U-235
true_pu241 = np.zeros((height, width))         # Pu-241 only in center
true_pu241[5:15, 5:15] = 0.0005

# Compute transmission for each pixel using forward_model
n_energies = len(energies)
transmission = np.zeros((n_energies, height, width))

for y in range(height):
    for x in range(width):
        d_u = true_u235[y, x]
        d_pu = true_pu241[y, x]
        isotopes = [(u235, d_u)]
        if d_pu > 0:
            isotopes.append((pu241, d_pu))
        transmission[:, y, x] = nereids.forward_model(
            energies, isotopes, temperature_k=293.6
        )

# Add Poisson-like noise (assume 1000 open-beam counts per bin)
flux = 1000.0
rng = np.random.default_rng(42)
noisy_counts = rng.poisson(flux * transmission)
noisy_transmission = noisy_counts / flux
# Poisson uncertainty: sigma_T = sqrt(counts) / flux
uncertainty = np.sqrt(np.maximum(noisy_counts, 1.0)) / flux

print(f"Transmission cube shape: {noisy_transmission.shape}")
print(f"Min/max transmission: {noisy_transmission.min():.3f} / {noisy_transmission.max():.3f}")

## 2. Visualize the Data

Let's look at a single energy frame and the ground-truth density maps.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Transmission at ~6.7 eV (near U-235 resonance)
e_idx = np.argmin(np.abs(energies - 6.7))
im0 = axes[0].imshow(noisy_transmission[e_idx], cmap='viridis', vmin=0, vmax=1)
axes[0].set_title(f'Transmission at {energies[e_idx]:.1f} eV')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Ground truth: U-235
im1 = axes[1].imshow(true_u235 * 1000, cmap='Blues')
axes[1].set_title('True U-235 density (×10³)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Ground truth: Pu-241
im2 = axes[2].imshow(true_pu241 * 1000, cmap='Reds')
axes[2].set_title('True Pu-241 density (×10³)')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## 3. Region-of-Interest Fitting

`fit_roi()` averages the transmission spectrum over a rectangular region,
giving a high-statistics single spectrum that can be fitted reliably.
This is useful as a quick check before running the full spatial map.

In [ ]:
# Fit the center region (where both isotopes are present)
roi_result = nereids.fit_roi(
    noisy_transmission, uncertainty,
    y_range=(5, 15), x_range=(5, 15),
    energies=energies,
    isotopes=[u235, pu241],
    temperature_k=293.6,
    initial_densities=[0.002, 0.002],
)

print(f"ROI fit result: {roi_result}")
print(f"  U-235:  fitted={roi_result.densities[0]:.6f}, true=0.001000")
print(f"  Pu-241: fitted={roi_result.densities[1]:.6f}, true=0.000500")
print(f"  U-235  error: {abs(roi_result.densities[0] - 0.001)/0.001*100:.1f}%")
print(f"  Pu-241 error: {abs(roi_result.densities[1] - 0.0005)/0.0005*100:.1f}%")

## 4. Full Spatial Mapping

`spatial_map()` fits every pixel independently in parallel using rayon.
Each pixel gets its own LM fit, recovering per-pixel areal densities.

In [ ]:
t0 = time.time()
result = nereids.spatial_map(
    noisy_transmission, uncertainty,
    energies=energies,
    isotopes=[u235, pu241],
    temperature_k=293.6,
    initial_densities=[0.002, 0.002],
)
elapsed = time.time() - t0

print(f"Spatial mapping: {result}")
print(f"Elapsed: {elapsed:.2f} s ({height*width} pixels, {elapsed/(height*width)*1000:.1f} ms/pixel)")

In [ ]:
# Plot fitted density maps side-by-side with ground truth
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# U-235: truth vs fitted
im00 = axes[0, 0].imshow(true_u235 * 1000, cmap='Blues', vmin=0, vmax=1.2)
axes[0, 0].set_title('U-235 Truth (×10³ atoms/barn)')
plt.colorbar(im00, ax=axes[0, 0], fraction=0.046)

fitted_u235 = np.array(result.density_maps[0])
im01 = axes[0, 1].imshow(fitted_u235 * 1000, cmap='Blues', vmin=0, vmax=1.2)
axes[0, 1].set_title('U-235 Fitted (×10³ atoms/barn)')
plt.colorbar(im01, ax=axes[0, 1], fraction=0.046)

# Pu-241: truth vs fitted
im10 = axes[1, 0].imshow(true_pu241 * 1000, cmap='Reds', vmin=0, vmax=0.6)
axes[1, 0].set_title('Pu-241 Truth (×10³ atoms/barn)')
plt.colorbar(im10, ax=axes[1, 0], fraction=0.046)

fitted_pu241 = np.array(result.density_maps[1])
im11 = axes[1, 1].imshow(fitted_pu241 * 1000, cmap='Reds', vmin=0, vmax=0.6)
axes[1, 1].set_title('Pu-241 Fitted (×10³ atoms/barn)')
plt.colorbar(im11, ax=axes[1, 1], fraction=0.046)

for ax in axes.flat:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.suptitle('Spatial Mapping: Ground Truth vs Fitted Density Maps', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic maps: convergence and reduced chi-squared
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(np.array(result.converged_map).astype(float), cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_title(f'Convergence Map ({result.n_converged}/{result.n_total} converged)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

chi2 = np.array(result.chi_squared_map)
im1 = axes[1].imshow(chi2, cmap='hot', vmin=0, vmax=np.nanpercentile(chi2, 95))
axes[1].set_title(f'Reduced Chi-Squared (median={np.nanmedian(chi2):.2f})')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## 5. Quantitative Comparison

How well did the fit recover the true densities?

In [ ]:
# Error statistics for converged pixels
converged = np.array(result.converged_map)

# U-235: present everywhere at 0.001
u235_err = (fitted_u235[converged] - true_u235[converged]) / true_u235[converged] * 100
print(f"U-235 relative error (converged pixels):")
print(f"  Mean:   {np.mean(u235_err):+.2f}%")
print(f"  Std:    {np.std(u235_err):.2f}%")
print(f"  Max:    {np.max(np.abs(u235_err)):.2f}%")

# Pu-241: present only in center at 0.0005
center_mask = converged & (true_pu241 > 0)
outer_mask = converged & (true_pu241 == 0)

pu241_center_err = (fitted_pu241[center_mask] - true_pu241[center_mask]) / true_pu241[center_mask] * 100
print(f"\nPu-241 center region (true=0.0005):")
print(f"  Mean:   {np.mean(pu241_center_err):+.2f}%")
print(f"  Std:    {np.std(pu241_center_err):.2f}%")

pu241_outer = fitted_pu241[outer_mask]
print(f"\nPu-241 outer region (true=0, should be ~0):")
print(f"  Mean:   {np.mean(pu241_outer):.6f} atoms/barn")
print(f"  Max:    {np.max(pu241_outer):.6f} atoms/barn")

## 6. Dead Pixel Masking

`spatial_map()` supports an optional `dead_pixels` argument: a 2D boolean
array where `True` marks pixels to skip. Dead pixels remain at zero in
the output maps.

In [ ]:
# Create a dead pixel mask (mark corners as dead)
dead = np.zeros((height, width), dtype=bool)
dead[0, 0] = True
dead[0, -1] = True
dead[-1, 0] = True
dead[-1, -1] = True

result_masked = nereids.spatial_map(
    noisy_transmission, uncertainty,
    energies=energies,
    isotopes=[u235, pu241],
    temperature_k=293.6,
    dead_pixels=dead,
)

print(f"With dead pixels: {result_masked}")
print(f"Density at dead pixel [0,0]: {result_masked.density_maps[0][0,0]:.6f} (expected 0)")

## 7. Loading Real TIFF Data (Optional)

The functions `load_tiff_stack()`, `normalize()`, and `tof_to_energy_centers()`
handle the full data loading pipeline for real VENUS beamline data.

This section demonstrates loading the PLEIADES synthetic TIFF stack if available.

In [ ]:
import os

TIFF_PATH = os.path.join('..', '..', '..', 'PLEIADES', 'tests', 'data',
                          'pleiades_data', 'LANL-ORNL_example.tif')
HAS_DATA = os.path.exists(TIFF_PATH)

if HAS_DATA:
    data = nereids.load_tiff_stack(TIFF_PATH)
    print(f"PLEIADES TIFF: shape={data.shape}, dtype={data.dtype}")
    print(f"  Value range: [{data.min():.3f}, {data.max():.3f}]")
    print(f"  This is pre-normalized transmission data (500 energy bins, 256×256 pixels)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, frame_idx in enumerate([50, 200, 400]):
        im = axes[i].imshow(data[frame_idx], cmap='viridis', vmin=0, vmax=1)
        axes[i].set_title(f'Frame {frame_idx}')
        plt.colorbar(im, ax=axes[i], fraction=0.046)
    plt.suptitle('PLEIADES Synthetic Data: Transmission Frames', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"PLEIADES data not found at {TIFF_PATH}")
    print("Skipping real data demo (this is expected in CI)")

In [ ]:
if HAS_DATA:
    # The normalize() function converts raw counts to transmission.
    # Since the PLEIADES data is already normalized, we demonstrate
    # the API with a small synthetic example instead.
    sample_raw = np.random.poisson(500, size=(10, 4, 4)).astype(float)
    ob_raw = np.random.poisson(1000, size=(10, 4, 4)).astype(float)

    trans_norm, unc_norm = nereids.normalize(sample_raw, ob_raw,
                                             pc_sample=1.5, pc_ob=1.0)
    print(f"normalize() output: trans shape={trans_norm.shape}, unc shape={unc_norm.shape}")
    print(f"  Mean transmission: {trans_norm.mean():.3f}")

    # tof_to_energy_centers: convert TOF bin edges to energy grid
    tof_edges = np.linspace(500, 20000, 501)  # 500 bins
    energy_centers = nereids.tof_to_energy_centers(tof_edges, flight_path_m=25.0)
    print(f"\ntof_to_energy_centers: {len(energy_centers)} centers")
    print(f"  Energy range: [{energy_centers[0]:.4f}, {energy_centers[-1]:.4f}] eV")

## Summary

This notebook demonstrated:

1. **Synthetic data generation** — forward model per pixel to create a test image
2. **`fit_roi()`** — high-statistics fit over a rectangular region
3. **`spatial_map()`** — parallel per-pixel fitting recovering 2D density maps
4. **Diagnostics** — convergence maps, chi-squared maps, error statistics
5. **Dead pixel masking** — skip bad pixels during fitting
6. **I/O functions** — `load_tiff_stack()`, `normalize()`, `tof_to_energy_centers()`

### Typical Workflow for Real Data

```python
# 1. Load data
sample = nereids.load_tiff_stack('sample.tif')
open_beam = nereids.load_tiff_stack('open_beam.tif')

# 2. Normalize
trans, unc = nereids.normalize(sample, open_beam, pc_sample=1.5, pc_ob=1.0)

# 3. Energy grid
tof_edges = np.loadtxt('tof_edges.txt')  # from instrument
energies = nereids.tof_to_energy_centers(tof_edges, flight_path_m=25.0)

# 4. Load isotopes
u235 = nereids.load_endf(92, 235)
pu241 = nereids.load_endf(94, 241)

# 5. Map!
result = nereids.spatial_map(trans, unc, energies, [u235, pu241])
```

**Previous**: [06_custom_resolution.ipynb](06_custom_resolution.ipynb)